# 03 — Huấn luyện E01–E04

Gọi `src/train.load_experiment_config` và `src/train.run_training` với một file trong `configs/experiments/`.
`run_training` tự chọn pipeline theo cấu hình:

| Thí nghiệm | Mô hình | Dữ liệu | Checkpoint |
|---|---|---|---|
| E01 | CNN tự thiết kế (Keras) + class weights | `v1_crop128` | `best.keras` |
| E02 | E01 + augmentation | `v1_crop128` | `best.keras` |
| E03 | MobileNetV2 ImageNet (PyTorch), 2 pha | `v1_crop128` | `best.pt` |
| E04 | EfficientNet-B0, 18 lớp, bù lớp hiếm, chọn checkpoint theo Macro F1 | `v2_crop224` | `best.pt` |

Checkpoint, `history.csv`, `config.json` lưu ở Drive `runs/<EXP_ID>_<YYYYMMDD-HHMM>/`.

**Trước khi chạy trên Colab:** Runtime → Change runtime type → **T4 GPU**. Không commit `*.keras`, `*.pt`.

> Notebook này **chỉ gọi hàm trong `src/`**: không `%%writefile`, không chép code nguồn vào cell, không token,
> không `git push`. Muốn sửa logic → sửa `src/` + `tests/` trên nhánh riêng và mở PR.

## 1. Cài đặt và dữ liệu

In [ ]:
# Cài đặt chung: Colab → mount Drive + clone repo; máy cá nhân → dùng repo đang mở.
import os, sys, subprocess, json, time
from pathlib import Path

BRANCH = "main"   # đổi thành "feat/s2-pipeline-end-to-end" nếu PR #21 chưa merge
REPO_URL = "https://github.com/phanyen06ym-ien/ocean-plastic-ml-starter.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/ocean-plastic-ml")   # thư mục chung của nhóm
    WORK = Path("/content")                                   # ổ đĩa Colab, đọc nhanh hơn Drive
    REPO = WORK / "repo"
    if not REPO.exists():
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    DRIVE = REPO                                              # data/, runs/ đã nằm trong .gitignore
    WORK = REPO / "data"
os.chdir(REPO); sys.path.insert(0, str(REPO))
for sub in ["data/raw", "data/processed", "runs"]:
    (DRIVE / sub).mkdir(parents=True, exist_ok=True)
print("Repo:", REPO, "| commit:",
      subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

In [ ]:
# Chọn thí nghiệm, rồi lấy đúng phiên bản dữ liệu của thí nghiệm đó (tạo bởi notebook 02).
from src.train import load_experiment_config, is_keras_config

EXP_CONFIG = "configs/experiments/E01_baseline.yaml"
# E02: "configs/experiments/E02_augment.yaml"
# E03: "configs/experiments/E03_transfer_mobilenetv2.yaml"
# E04: "configs/experiments/E04_efficientnet_b0_data_v2.yaml"
cfg = load_experiment_config(REPO / EXP_CONFIG)
DATA_VERSION = cfg["data_version"] if is_keras_config(cfg) else cfg["data"]["data_version"]
PROCESSED = WORK / "processed" / DATA_VERSION
PROCESSED_ZIP = DRIVE / "data/processed" / f"{DATA_VERSION}.zip"
if IN_COLAB and not PROCESSED.exists() and PROCESSED_ZIP.exists():
    PROCESSED.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["unzip", "-q", str(PROCESSED_ZIP), "-d", str(PROCESSED.parent)], check=True)
assert (PROCESSED / "split_manifest.csv").exists(), f"Chưa có {PROCESSED} — chạy notebook 02 với đúng data contract trước"
print(cfg["exp_id"], "| dữ liệu:", PROCESSED)

In [ ]:
if is_keras_config(cfg):
    import tensorflow as tf
    print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))
else:
    import torch
    print("PyTorch", torch.__version__, "| GPU:", torch.cuda.is_available())

## 2. Cấu hình thí nghiệm

In [ ]:
print(json.dumps(cfg, ensure_ascii=False, indent=2))
if is_keras_config(cfg):
    from src.train import model_from_config
    model_from_config(cfg).summary()
else:
    from src.train import build_model, count_params
    print(count_params(build_model(cfg, pretrained=False)))

## 3. Huấn luyện

In [ ]:
from src.train import run_training

RUNS = DRIVE / "runs"
t0 = time.time()
if is_keras_config(cfg):
    RESUME_RUN_DIR = None   # Colab ngắt giữa chừng → đặt RUNS / "E01_<YYYYMMDD-HHMM>" để chạy tiếp từ last.keras
    model, history, RUN_DIR = run_training(cfg, PROCESSED, RUNS, run_dir=RESUME_RUN_DIR, resume=RESUME_RUN_DIR is not None)
else:
    model, history, RUN_DIR = run_training(cfg, PROCESSED, RUNS)
print(f"{cfg['exp_id']} xong sau {(time.time() - t0) / 60:.1f} phút → {RUN_DIR}")

## 4. Đường cong học tập (Train / Validation)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

h = pd.read_csv(RUN_DIR / "history.csv")
best = int(h["val_loss"].idxmin())
print(f"Epoch tốt nhất theo val_loss: {best + 1} | val_loss={h['val_loss'][best]:.4f}"
      f" | val_accuracy={h['val_accuracy'][best]:.4f}")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(h["loss"], label="train"); axes[0].plot(h["val_loss"], label="val"); axes[0].set_title(f"{cfg['exp_id']}: loss")
axes[1].plot(h["val_accuracy"], label="val accuracy"); axes[1].set_title(f"{cfg['exp_id']}: val accuracy")
for ax in axes:
    ax.set_xlabel("epoch"); ax.legend()
fig.tight_layout(); display(fig); plt.close(fig)
print("Đánh giá bằng notebook 04:", RUN_DIR)